In [41]:
from sympy import *
from sympy.assumptions import Q
from sympy.assumptions.refine import refine
import os
# import jax
# jax.config.update("jax_enable_x64", True)

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.5"

alpha, beta, x_t, x_t1, x, theta, T_t, T_t1 = symbols(r'alpha beta x_t x_{t+1} x \theta T_t, T_{t+1}') # 変数
sita_0, sita_pre, sita_aft, time_pre, time_aft = symbols(r'\theta_{0} \theta_{pre} \theta_{aft} T_{pre} T_{aft}') # 変数

In [42]:
# 近日点から0~πの移動面積を計算する関数を作成

f = x**2/alpha**2 + (x*tan(theta)-sqrt(1-alpha**2))**2 -1

f = collect(expand(f), x**2)

x_s = solve(f,x)
X = x_s[1]

A=latex((x_s[0]))

Y = tan(theta)*X-sqrt(1-alpha**2)
Y=simplify(Y)

f = refine(Y/X, Q.positive(cos(theta)))
f = simplify(f)

X = simplify(refine(X, Q.positive(cos(theta))))

tri = sqrt(1-alpha**2)*X/2
arc = (alpha/2)*atan(alpha*f)

pSS = arc - tri

pft_fun = pSS.subs({theta:sita_0+sita_aft})*time_pre/time_aft - pSS.subs({theta:sita_0+sita_pre}) + pSS.subs({theta:sita_0})*(1 - time_pre/time_aft)

pft_xi = diff(pft_fun, sita_pre)
pft_xN = diff(pft_fun, sita_aft)
pft_a = diff(pft_fun, alpha)
pft_b = diff(pft_fun, sita_0)

pg_xN = -pft_xN/pft_xi
pg_a = -pft_a/pft_xi
pg_b = -pft_b/pft_xi


In [43]:
# 近日点からπ~2πの移動面積を計算する関数を作成

f = x**2/alpha**2 + (x*tan(theta)-sqrt(1-alpha**2))**2 -1
# display(f)

f = collect(expand(f), x**2)

x_s = solve(f,x)
X = x_s[0]

Y = X*tan(theta) - sqrt(1-alpha**2)
Y = simplify(Y)

f = refine(Y/X, Q.negative(cos(theta)))
f = simplify(f)

X = simplify(refine(X, Q.negative(cos(theta))))

tri = sqrt(1-alpha**2)*X/2
arc = (alpha/2)*(atan(alpha*f)+pi)

aSS = arc - tri

aft_fun = pSS.subs({theta:sita_0+sita_aft})*(time_pre/time_aft) - aSS.subs({theta:sita_0+sita_pre}) + pSS.subs({theta:sita_0})*(1 - time_pre/time_aft)

aft_xi = diff(aft_fun, sita_pre)
aft_xN = diff(aft_fun, sita_aft)
aft_a = diff(aft_fun, alpha)
aft_b = diff(aft_fun, sita_0)

ag_xN = -aft_xN/aft_xi
ag_a = -aft_a/aft_xi
ag_b = -aft_b/aft_xi


In [44]:
import numpy as np
import plotly.graph_objects as go
from sympy import lambdify
# alpha を固定
alpha_val = 0.5

# θ ∈ [π/2, π]
theta_grid_p = np.linspace(
    - np.pi / 2 + 0.00000001,
    np.pi / 2 - 0.00000001,
    1000
)

# θ ∈ [π/2, π]
theta_grid_a = np.linspace(
    np.pi / 2 + 0.00000001,
    3*np.pi / 2 - 0.00000001,
    1000
)

pSS_func = lambdify((theta, alpha), tri, "numpy")
aSS_func = lambdify((theta, alpha), tri, "numpy")

y_p = pSS_func(theta_grid_p, alpha_val)
y_a = aSS_func(theta_grid_a, alpha_val)

y = np.concatenate([y_p, y_a])
theta_grid = np.concatenate([theta_grid_p, theta_grid_a])

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=theta_grid,
        y=y,
        mode="lines",
        name="面積"
    )
)

fig.update_layout(
    title=f"θと地球の移動面積の関係 (alpha={alpha_val})",
    xaxis_title="theta [rad]",
    yaxis_title="aSS",
)

fig.show()

## 観測データ作成

In [45]:
import numpy as np
from skyfield.api import load
from datetime import datetime, timedelta
import math


# =========================
# ベクトル角度（cosθ）
# =========================
def angle_between(a, b):

    dot = sum(x * y for x, y in zip(a, b))

    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(x * x for x in b))

    if norm_a == 0 or norm_b == 0:
        raise ValueError("ゼロベクトルは角度を定義できません")

    cos_theta = dot / (norm_a * norm_b)

    cos_theta = max(-1.0, min(1.0, cos_theta))

    return cos_theta


# =========================
# Skyfield
# =========================
ts = load.timescale()

planets = load("de421.bsp")

sun = planets["sun"]
earth = planets["earth"]


# =========================
# 日付生成（観測期間のみ）
# =========================
periods = [
    (
        datetime(2025, 1, 15, 20, 0, 0),
        datetime(2025, 6, 16, 20, 0, 0),
        True,
    ),
    (
        datetime(2025, 7, 10, 20, 0, 0),
        datetime(2025, 12, 27, 20, 0, 0),
        False,
    ),
]

dates = []
isP = []

for start_date, end_date, flag in periods:

    d = start_date

    while d <= end_date:

        dates.append(
            ts.utc(
                d.year,
                d.month,
                d.day,
                20,
                0,
                0,
            )
        )

        isP.append(flag)

        d += timedelta(days=1)


# =========================
# 地球位置
# =========================
positions = []

for t in dates:

    pos = (earth - sun).at(t).position.km

    positions.append(pos)


# =========================
# 公転角 Y
# Y[i-1] > Y[i] なら
# [π, 2π] 側へ再計算
# =========================
Y = []
T = []

for i in range(1, len(positions)):

    cos_theta = angle_between(
        positions[0],
        positions[i - 1],
    )

    # 通常の arccos: [0, π]
    theta = np.arccos(cos_theta)

    # 必要なら [π, 2π] に切り替え
    if len(Y) > 0 and theta < Y[-1]:
        theta = 2 * np.pi - theta

    Y.append(theta)

    dt_days = dates[i].tt - dates[i - 1].tt
    T.append(dt_days)


# =========================
# numpy array化
# =========================
isP = np.array(isP)

Yo = np.array(Y)

sigma = 0.0001
sigma = 0.01

Y = Yo + np.random.normal(
    0,
    sigma,
    size=Yo.shape,
)

T = np.array(T)
Y1 = Y.copy()

#Y = gaussian_smooth(Y)
T = T[0:len(isP)-2]
To = T.copy()
T = np.cumsum(T)
isP = isP[1:len(isP)-1]
Y = Y[1:len(Y)]

num_of_p = np.sum(isP) -1


In [46]:
t = np.arange(len(Y))

fig = go.Figure()

Yo = Yo[1:]
fig.add_trace(
    go.Scatter(
        x=t,
        y=np.abs(Yo-Y),
        mode='lines',
        name='Prediction X'
    )
)

fig.update_layout(
    title='|真値-観測値|',
    xaxis_title="Time [day]",
    yaxis_title="θ [rad]",
    template='plotly_white'
)
print(np.average(np.abs(Yo-Y)))
fig.show()

0.008061589367468527


In [47]:
pm_ft = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft),pft_fun, "mpmath")
am_ft = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft),aft_fun, "mpmath")

pmg_a  = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), pg_a,  "mpmath")
pmg_xN = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), pg_xN, "mpmath")
pmg_b  = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), pg_b,  "mpmath")

amg_a  = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), ag_a,  "mpmath")
amg_xN = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), ag_xN, "mpmath")
amg_b  = lambdify((alpha, sita_0, sita_pre, sita_aft, time_pre, time_aft), ag_b,  "mpmath")

In [48]:
import numpy as np
from scipy.optimize import minimize, root_scalar
import mpmath as mp
import warnings

def calc_y(a, b, sa, tp, ti, x0, isp = True):
    def F(x):
        if isp == False and b+x < np.pi/2:
            return np.pi
        
        if isp:
            return float(pm_ft(mp.mpf(a), mp.mpf(b), mp.mpf(x), mp.mpf(sa), mp.mpf(tp), mp.mpf(ti)))
        else:
            return float(am_ft(mp.mpf(a), mp.mpf(b), mp.mpf(x), mp.mpf(sa), mp.mpf(tp), mp.mpf(ti)))
    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore",
            category=RuntimeWarning,
            message=r"Tolerance of .* reached\."
        )

        result = root_scalar(
            F,
            x0=x0,
            x1=x0 + 1e-3,
            method="secant",
            xtol=1e-14
        )
    return result.root

T = np.array(T)
Y = np.array(Y)

Tlen = len(T)

def objective(X, ret_div = True):
    a = X[0] if X[0] < 0.99999999999 else 0.99999999999
    b = X[1]
    yl = X[2]
    grad = [mp.mpf(0.0),mp.mpf(0.0),mp.mpf(0.0)]

    Xt = []
    for i in range(Tlen): #  
        if i == 0:
            y = calc_y(a, b, yl, T[i], T[num_of_p], 0.0, isP[i])
        elif isP[i - 1] == True and isP[i] == False:
            y = calc_y(a, b, yl, T[i], T[num_of_p], Y[num_of_p+1], isP[i])
        else:
            y = calc_y(a, b, yl, T[i], T[num_of_p], Xt[i-1], isP[i])
        Xt.append(y)
        if isP[i]:
            ig = np.array(
                [
                    mp.mpf(Y[i] - y)*pmg_a(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                    mp.mpf(Y[i] - y)*pmg_b(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                    mp.mpf(Y[i] - y)*pmg_xN(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                ]
            )
        else:
            ig = np.array(
                [
                    mp.mpf(Y[i] - y)*amg_a(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                    mp.mpf(Y[i] - y)*amg_b(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                    mp.mpf(Y[i] - y)*amg_xN(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                ]
            )
        grad[0] += ig[0]
        grad[1] += ig[1]
        grad[2] += ig[2]

    grad[0] = -2*grad[0]
    grad[1] = -2*grad[1]
    grad[2] = -2*grad[2]
    
    grad = np.array([float(grad[0]), float(grad[1]), float(grad[2])])

    Xt = np.array(Xt)
    val = np.sum((Y - Xt)**2)

    n = np.linalg.norm(grad)

    if ret_div == False:
        return np.nan_to_num(
            val,
            nan=1e30,
            posinf=1e30,
            neginf=1e30
        )

    return np.nan_to_num(
        val,
        nan=1e30,
        posinf=1e30,
        neginf=1e30
    ), grad


# 初期値
X0 = np.concatenate([np.array([0.99999999999, -np.pi/2, Y[num_of_p]])])

result = minimize(
    objective,
    X0,
    method="L-BFGS-B",
    jac=True,
    bounds=[
        (1e-6, 0.99999999999),   # alpha
        (np.pi/2 - Y[num_of_p+1], np.pi/2 - Y[num_of_p] ),       # beta
        (None, None),       # xN
    ],
    options={
        "maxiter": 100000,
        "ftol": 0.0,
        "gtol": 1e-10,
        "maxls": 15,
        "maxcor": 30,
    }
)

print(result.x)
print(result)


[ 0.99985511 -1.35708631  2.61832895]
  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: 0.03270261386507363
        x: [ 9.999e-01 -1.357e+00  2.618e+00]
      nit: 39
      jac: [-4.757e-09  3.906e-11 -1.080e-10]
     nfev: 52
     njev: 52
 hess_inv: <3x3 LbfgsInvHessProduct with dtype=float64>


In [49]:
a = result.x[0]
b = result.x[1]
yl = result.x[2]

Xt = []
for i in range(Tlen): #  
    if i == 0:
        y = calc_y(a, b, yl, T[i], T[num_of_p], 0.0, isP[i])
    elif isP[i - 1] == True and isP[i] == False:
        y = calc_y(a, b, yl, T[i], T[num_of_p], np.pi+0.001, isP[i])
    else:
        y = calc_y(a, b, yl, T[i], T[num_of_p], Xt[i-1], isP[i])
    Xt.append(y)


import numpy as np
import plotly.graph_objects as go
from sympy import lambdify

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=T,
        y=Y,
        mode="lines",
        name="観測値"
    )
)

fig.add_trace(
    go.Scatter(
        x=T,
        y=Xt,
        mode="lines",
        name="予測値"
    )
)

fig.add_trace(
    go.Scatter(
        x=T,
        y=Yo,
        mode="lines",
        name="真値"
    )
)
fig.update_layout(
    xaxis_title="Time [day]",
    yaxis_title="θ [rad]",
)

fig.show()


In [50]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=T,
        y=(Xt-Yo),
        mode="lines",
        name="予測値との差"
    )
)

fig.add_trace(
    go.Scatter(
        x=T,
        y=(Yo-Y),
        mode="lines",
        name="観測値との差"
    )
)
fig.update_layout(
    xaxis_title="Time [day]",
    yaxis_title="θ [rad]",

)

fig.show()


# 誤差推定

フィッシャー行列をモンテカルロ法を使用して求め誤差推定を行う

In [51]:
fis = [mp.mpf(0.0), mp.mpf(0.0), mp.mpf(0.0), mp.mpf(0.0), mp.mpf(0.0), mp.mpf(0.0)]


for i in range(Tlen): #  
    if i == 0:
        y = calc_y(a, b, yl, T[i], T[num_of_p], 0.0, isP[i])
    elif isP[i - 1] == True and isP[i] == False:
        y = calc_y(a, b, yl, T[i], T[num_of_p], Y[num_of_p+1], isP[i])
    else:
        y = calc_y(a, b, yl, T[i], T[num_of_p], Xt[i-1], isP[i])
    if isP[i]:
        ig = np.array(
            [
                mp.mpf(Y[i] - y)*pmg_a(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                mp.mpf(Y[i] - y)*pmg_b(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                mp.mpf(Y[i] - y)*pmg_xN(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
            ]
        )
    else:
        ig = np.array(
            [
                mp.mpf(Y[i] - y)*amg_a(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                mp.mpf(Y[i] - y)*amg_b(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
                mp.mpf(Y[i] - y)*amg_xN(mp.mpf(a), mp.mpf(b), mp.mpf(y), mp.mpf(yl), mp.mpf(T[i]), mp.mpf(T[num_of_p])),
            ]
        )

    fis[0] += ig[0]*ig[0]
    fis[1] += ig[0]*ig[1]
    fis[2] += ig[0]*ig[2]
    fis[3] += ig[1]*ig[1]
    fis[4] += ig[1]*ig[2]
    fis[5] += ig[2]*ig[2]


F = np.array(
    [
        [float(fis[0]), float(fis[1]), float(fis[2])],
        [float(fis[1]), float(fis[3]), float(fis[4])],
        [float(fis[2]), float(fis[4]), float(fis[5])]
    ])/sigma**2


F_inv = np.linalg.inv(F)

print(F_inv)



[[ 7.92349762e-07 -8.45766758e-04 -7.46055740e-06]
 [-8.45766758e-04  5.14376101e+00 -1.86774573e-01]
 [-7.46055740e-06 -1.86774573e-01  1.05342933e-02]]


In [52]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

# 平均と分散
mu = result.x[0]
sigma2 = F_inv[0,0]

# 標準偏差
sigmau = np.sqrt(sigma2)

# 描画範囲（±5σ）
x = np.linspace(mu - 5*sigmau, mu + 5*sigmau, 1000)

# 正規分布の確率密度関数
y = norm.pdf(x, loc=mu, scale=sigmau)/np.sqrt(2*np.pi*sigmau)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=x,
        y=y,
        mode="lines",
        name=f"N({mu:.6f}, {sigma2:.6e})"
    )
)

fig.update_layout(
    title="Normal Distribution",
    xaxis_title="Parameter",
    yaxis_title="Probability Density",
)

fig.show()